In [1]:
import sys

sys.path.insert(0, "/path/to/your/relativistic")

from TDECalculator import TDECalculator
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from matplotlib.colors import LogNorm

plt.style.use("plot_style.mplstyle")
plt.rc('text', usetex=True)

In [2]:
calc_mams_pro = TDECalculator('MAMS3Msun_z0p0142', orbit='rel', MBH=1e6, Rp=18.2576, a=0.2, E=1.0, Q=0.0, N=5000)
calc_mams_sch = TDECalculator('MAMS3Msun_z0p0142', orbit='rel', MBH=1e6, Rp=18.2576, a=-0.0, E=1.0, Q=0.0, N=5000)
calc_mams_retro = TDECalculator('MAMS3Msun_z0p0142', orbit='rel', MBH=1e6, Rp=18.2576, a=0.2, E=1.0, Q=0.0, N=5000)  

In [3]:
calc_tams_pro = TDECalculator('TAMS3Msun_z0p0142', orbit='rel', MBH=1e6, Rp=23.94, a=0.2, E=1.0, Q=0.0, N=5000)
calc_tams_sch = TDECalculator('TAMS3Msun_z0p0142', orbit='rel', MBH=1e6, Rp=23.94, a=0.0, E=1.0, Q=0.0, N=5000)
calc_tams_retro = TDECalculator('TAMS3Msun_z0p0142', orbit='rel', MBH=1e6, Rp=23.94, a=-0.2, E=1.0, Q=0.0, N=5000) 

In [6]:
calc_rgb_pro = TDECalculator('rgb_3m_z0p0142_logR1p0', orbit='rel', MBH=1e6, Rp=66.514, a=0.2, E=1.0, Q=0.0, N=5000)        # prograde trajectory
calc_rgb_sch = TDECalculator('rgb_3m_z0p0142_logR1p0', orbit='rel', MBH=1e6, Rp=66.514, a=0.0, E=1.0, Q=0.0, N=5000)
calc_rgb_retro = TDECalculator('rgb_3m_z0p0142_logR1p0', orbit='rel', MBH=1e6, Rp=66.514, a=-0.2, E=1.0, Q=0.0, N=5000)

In [7]:
def density_panel(ax, calc):
    # Use equatorial slice
    equator_sample = calc.equator_sample()

    POS   = equator_sample['POS']
    dMass = equator_sample['dMass']

    # Flatten x, y, and mass
    x_pos = POS[..., 0].flatten()
    y_pos = POS[..., 1].flatten()
    mass_flat = dMass.flatten()

    # Histogram bins
    nbins = 300
    x_edges = np.linspace(-2, 2, nbins + 1)
    y_edges = np.linspace(-3, 3, nbins + 1)

    # 2D mass histogram
    mass_hist, x_edges, y_edges = np.histogram2d(
        x_pos,
        y_pos,
        bins=[x_edges, y_edges],
        weights=mass_flat
    )

    # Surface density
    dx = x_edges[1] - x_edges[0]
    dy = y_edges[1] - y_edges[0]
    density2D = mass_hist / (dx * dy)

    # Grid
    Xc, Yc = np.meshgrid(
        0.5 * (x_edges[:-1] + x_edges[1:]),
        0.5 * (y_edges[:-1] + y_edges[1:])
    )

    # Tiny offset for stable contours
    density2D += 1e-12 * (Xc + Yc)

    # Interpolation
    points = np.column_stack((Xc.flatten(), Yc.flatten()))
    values = density2D.T.flatten()

    xi = np.linspace(Xc.min(), Xc.max(), nbins)
    yi = np.linspace(Yc.min(), Yc.max(), nbins)

    XI, YI = np.meshgrid(xi, yi)

    dens_interp = griddata(
        points,
        values,
        (XI, YI),
        method='linear'
    )

    # Normalize to central density
    dens_interp += dens_interp.max() * 1e-8

    center_idx_x = XI.shape[0] // 2
    center_idx_y = XI.shape[1] // 2

    Sigma_c = dens_interp[center_idx_x, center_idx_y]
    dens_interp /= Sigma_c

    # Plot
    levels = np.logspace(
        np.log10(dens_interp[dens_interp > 0].min()),
        np.log10(dens_interp.max()),
        100
    )

    cf = ax.contourf(
        XI,
        YI,
        dens_interp,
        levels=levels,
        cmap='viridis',
        norm=LogNorm()
    )

    # Direction toward BH
    phi0 = calc.Psi_TDE if calc.orbit == "rel" else calc.Phi_TDE

    ax.arrow(
        -1.5, 2.25,
        -0.3 * np.cos(phi0),
        -0.3 * np.sin(phi0),
        linewidth=2,
        head_width=0.1,
        head_length=0.1,
        color='white'
    )

    ax.text(
        -1.5 - 0.3 * np.cos(phi0) + 0.1,
        2.25 - 0.3 * np.sin(phi0) + 0.1,
        'to BH',
        color='white',
        fontsize=10
    )

    # β
    beta = calc.Rtidal / calc.Rp

    ax.text(
        0.72, 0.92,
        fr'$\beta={beta:.2f}$',
        transform=ax.transAxes,
        color='white',
        fontsize=12,
        ha='left',
        va='top'
)

    ax.set_aspect('equal')
    ax.set_xlim(-2, 2)
    ax.set_ylim(-3, 3)
    ax.axis('off')

    return cf

In [8]:
calculators = [
    [calc_mams_pro, calc_mams_sch, calc_mams_retro],
    [calc_tams_pro, calc_tams_sch, calc_tams_retro],
    [calc_rgb_pro,  calc_rgb_sch,  calc_rgb_retro]
]

row_labels = ["MAMS", "TAMS", "RGB"]
column_labels = ["Prograde", "Schwarzschild", "Retrograde"]

calculators = [
    [calc_mams_pro, calc_mams_sch, calc_mams_retro],
    [calc_tams_pro, calc_tams_sch, calc_tams_retro],
    [calc_rgb_pro, calc_rgb_sch, calc_rgb_retro]
]

In [ ]:
fig, axes = plt.subplots(
    3, 3,
    figsize=(17, 15),
    gridspec_kw={
        "wspace": 0.0,
        "hspace": 0.03
    }
)

cf = None

for i in range(3):
    for j in range(3):

        calc = calculators[i][j]

        panel_cf = density_panel(axes[i, j], calc)

        if cf is None:
            cf = panel_cf

        # Column labels
        if i == 0:
            axes[i, j].set_title(
                column_labels[j],
                fontsize=25,
                fontweight="bold",
                pad=8
            )

        # Row labels
        if j == 0:
            axes[i, j].text(
                -0.12, 0.5,
                row_labels[i],
                transform=axes[i, j].transAxes,
                fontsize=18,
                rotation=90,
                va="center",
                ha="center"
            )

fig.subplots_adjust(
    left=0.05,
    right=0.91,
    bottom=0.025,
    top=0.96,
    wspace=0.0,
    hspace=0.03
)

cbar_ax = fig.add_axes([
    0.925,
    0.15,
    0.02,
    0.70
])

fig.colorbar(
    cf,
    cax=cbar_ax,
    label=r'$\rho/\rho_c$',
    ticks=[1e-8, 1e-6, 1e-4, 1e-2, 1]
)

plt.show()